In [2]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from scipy.spatial.distance import cosine
from datetime import datetime
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])

X_train_raw = train_df[spectrum_cols].to_numpy(dtype=float)
y_train = train_df[target_col].values
species_train = train_df["species number"].values
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=float)

print(f"  訓練: {X_train_raw.shape}, テスト: {X_test_raw.shape}")

# ============================================================
# 2. 前処理関数（テストにも安全に適用できるもののみ）
# ============================================================
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return (X - m) / s

def sg(X, window=17, poly=2, deriv=0):
    return savgol_filter(X, window, poly, deriv=deriv, axis=1)

def prep_raw(X_tr, X_te):
    return X_tr.copy(), X_te.copy()

def prep_snv(X_tr, X_te):
    return snv(X_tr), snv(X_te)

def prep_snv_sg1d(X_tr, X_te):
    return sg(snv(X_tr), 17, 2, 1), sg(snv(X_te), 17, 2, 1)

def prep_sg1d(X_tr, X_te):
    return sg(X_tr, 17, 2, 1), sg(X_te, 17, 2, 1)

def prep_snv_sg2d(X_tr, X_te):
    return sg(snv(X_tr), 21, 3, 2), sg(snv(X_te), 21, 3, 2)

PREPS = {
    "raw":       prep_raw,
    "SNV":       prep_snv,
    "SNV+SG1d":  prep_snv_sg1d,
    "SG1d":      prep_sg1d,
    "SNV+SG2d":  prep_snv_sg2d,
}

# ============================================================
# 3. 特徴量生成関数
# ============================================================
def feat_full_spectrum(X_tr, X_te, y_tr, wn):
    """スペクトル全波数をそのまま使用"""
    return X_tr, X_te, "full_spectrum"

def feat_pca(X_tr, X_te, y_tr, wn, n_components=20):
    """PCAで次元削減"""
    pca = PCA(n_components=n_components)
    F_tr = pca.fit_transform(X_tr)
    F_te = pca.transform(X_te)
    return F_tr, F_te, f"PCA({n_components})"

def feat_water_bands(X_tr_raw, X_te_raw, y_tr, wn):
    """水の吸収帯から物理的特徴量を抽出（rawスペクトルから計算）"""
    def extract(X):
        bands = {
            "OH_comb":  (5050, 5350),
            "OH_1st":   (6700, 7200),
            "OH_2nd":   (8200, 8800),
            "CH_1st":   (5600, 6000),
            "CH_comb":  (4000, 4500),
        }
        feats = {}
        for name, (lo, hi) in bands.items():
            mask = (wn >= lo) & (wn <= hi)
            if mask.sum() > 0:
                region = X[:, mask]
                feats[f"{name}_mean"] = region.mean(axis=1)
                feats[f"{name}_max"] = region.max(axis=1)
                feats[f"{name}_area"] = region.sum(axis=1)
        # バンド比（散乱に強い）
        if "OH_comb_mean" in feats and "CH_1st_mean" in feats:
            feats["OH_comb_div_CH"] = feats["OH_comb_mean"] / (feats["CH_1st_mean"] + 1e-8)
            feats["OH_1st_div_CH"] = feats["OH_1st_mean"] / (feats["CH_1st_mean"] + 1e-8)
        return np.column_stack(list(feats.values()))

    return extract(X_tr_raw), extract(X_te_raw), "water_bands"

def feat_moisture_similarity(X_tr, X_te, y_tr, wn, n_bins=5):
    """含水率帯ごとの平均スペクトルとのコサイン類似度"""
    bins = np.linspace(y_tr.min(), y_tr.max() + 1, n_bins + 1)
    ref_spectra = []
    for i in range(n_bins):
        mask = (y_tr >= bins[i]) & (y_tr < bins[i + 1])
        if mask.sum() > 0:
            ref_spectra.append(X_tr[mask].mean(axis=0))
        else:
            ref_spectra.append(np.zeros(X_tr.shape[1]))
    ref_spectra = np.array(ref_spectra)

    def calc_sim(X):
        sims = np.zeros((X.shape[0], n_bins))
        for i in range(n_bins):
            ref_norm = np.linalg.norm(ref_spectra[i])
            if ref_norm > 0:
                for j in range(X.shape[0]):
                    x_norm = np.linalg.norm(X[j])
                    if x_norm > 0:
                        sims[j, i] = np.dot(X[j], ref_spectra[i]) / (x_norm * ref_norm)
        return sims

    return calc_sim(X_tr), calc_sim(X_te), f"moisture_sim({n_bins})"

def feat_combined_compact(X_tr, X_te, y_tr, wn, X_tr_raw, X_te_raw, n_pca=20, n_bins=5):
    """案C: PCAスコア + 水吸収帯 + 含水率類似度"""
    # PCA
    pca = PCA(n_components=n_pca)
    F_pca_tr = pca.fit_transform(X_tr)
    F_pca_te = pca.transform(X_te)

    # 水吸収帯（rawから）
    F_water_tr, F_water_te, _ = feat_water_bands(X_tr_raw, X_te_raw, y_tr, wn)

    # 含水率帯類似度（前処理後スペクトルから）
    F_sim_tr, F_sim_te, _ = feat_moisture_similarity(X_tr, X_te, y_tr, wn, n_bins)

    F_tr = np.hstack([F_pca_tr, F_water_tr, F_sim_tr])
    F_te = np.hstack([F_pca_te, F_water_te, F_sim_te])
    return F_tr, F_te, f"combined(pca{n_pca}+water+sim{n_bins})"

# ============================================================
# 4. GroupKFold 評価関数
# ============================================================
N_SPLITS = 5

def evaluate(X_tr_all, y_all, species_all, model_fn,
             X_tr_raw_all=None, X_te_raw_all=None,
             prep_fn=None, feat_fn=None, feat_kwargs=None):
    """
    GroupKFoldで評価。
    前処理・特徴量生成もfold内で行う（リーク防止）。
    """
    cv = GroupKFold(n_splits=N_SPLITS)
    oof = np.full(len(y_all), np.nan)
    fold_rmses = []

    for tr_idx, va_idx in cv.split(X_tr_all, y_all, groups=species_all):
        # 分割
        Xr_tr, Xr_va = X_tr_all[tr_idx], X_tr_all[va_idx]
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        # rawも分割（水吸収帯特徴量用）
        if X_tr_raw_all is not None:
            Xraw_tr = X_tr_raw_all[tr_idx]
            Xraw_va = X_tr_raw_all[va_idx]
        else:
            Xraw_tr, Xraw_va = Xr_tr, Xr_va

        # 前処理
        if prep_fn is not None:
            Xp_tr, Xp_va = prep_fn(Xr_tr, Xr_va)
        else:
            Xp_tr, Xp_va = Xr_tr.copy(), Xr_va.copy()

        # 特徴量生成
        if feat_fn is not None:
            kwargs = feat_kwargs or {}
            if "X_tr_raw" in feat_fn.__code__.co_varnames:
                F_tr, F_va, _ = feat_fn(Xp_tr, Xp_va, y_tr, wavenumbers,
                                        Xraw_tr, Xraw_va, **kwargs)
            else:
                F_tr, F_va, _ = feat_fn(Xp_tr, Xp_va, y_tr, wavenumbers, **kwargs)
        else:
            F_tr, F_va = Xp_tr, Xp_va

        # モデル学習・予測
        model = model_fn()
        if isinstance(model, CatBoostRegressor):
            model.fit(F_tr, y_tr, eval_set=(F_va, y_va), verbose=0)
        elif isinstance(model, LGBMRegressor):
            model.fit(F_tr, y_tr, eval_set=[(F_va, y_va)],
                      callbacks=[early_stopping(50), log_evaluation(0)])
        elif isinstance(model, PLSRegression):
            model.fit(F_tr, y_tr)
        else:
            model.fit(F_tr, y_tr)

        preds = model.predict(F_va).ravel()
        oof[va_idx] = preds
        fold_rmses.append(np.sqrt(mean_squared_error(y_va, preds)))

    valid = ~np.isnan(oof)
    overall_rmse = np.sqrt(mean_squared_error(y_all[valid], oof[valid]))
    return overall_rmse, fold_rmses, oof

# ============================================================
# 5. 実験定義
# ============================================================
experiments = []

# --- 案A: PLS回帰（成分数を振る）---
for prep_name in ["SNV", "SNV+SG1d", "SG1d"]:
    for n_comp in [2, 3, 5, 7, 10, 15]:
        experiments.append({
            "name": f"A_PLS({n_comp})__{prep_name}",
            "group": "A: PLS",
            "prep_name": prep_name,
            "model_fn": lambda nc=n_comp: PLSRegression(n_components=nc),
            "feat_fn": None,
            "feat_kwargs": {},
        })

# --- 案A追加: Ridge回帰（正則化強度を振る）---
for prep_name in ["SNV", "SNV+SG1d"]:
    for alpha in [0.1, 1, 10, 100, 1000]:
        experiments.append({
            "name": f"A_Ridge(a={alpha})__{prep_name}",
            "group": "A: Ridge",
            "prep_name": prep_name,
            "model_fn": lambda a=alpha: Ridge(alpha=a),
            "feat_fn": None,
            "feat_kwargs": {},
        })

# --- 案B: PCA + モデル ---
for prep_name in ["SNV", "SNV+SG1d"]:
    for n_pca in [5, 10, 20, 30, 50]:
        # PCA + Ridge
        experiments.append({
            "name": f"B_PCA({n_pca})+Ridge__{prep_name}",
            "group": "B: PCA+Model",
            "prep_name": prep_name,
            "model_fn": lambda: Ridge(alpha=10),
            "feat_fn": feat_pca,
            "feat_kwargs": {"n_components": n_pca},
        })
        # PCA + LightGBM（少数特徴量なので過学習しにくい）
        experiments.append({
            "name": f"B_PCA({n_pca})+LGB__{prep_name}",
            "group": "B: PCA+Model",
            "prep_name": prep_name,
            "model_fn": lambda: LGBMRegressor(
                n_estimators=500, learning_rate=0.05, num_leaves=15,
                min_child_samples=20, verbosity=-1, random_state=42),
            "feat_fn": feat_pca,
            "feat_kwargs": {"n_components": n_pca},
        })

# --- 案B追加: KNN（外挿にならない直接的な方法）---
for prep_name in ["SNV", "SNV+SG1d"]:
    for n_pca in [10, 20]:
        for k in [3, 5, 10]:
            experiments.append({
                "name": f"B_PCA({n_pca})+KNN(k={k})__{prep_name}",
                "group": "B: KNN",
                "prep_name": prep_name,
                "model_fn": lambda k_=k: KNeighborsRegressor(
                    n_neighbors=k_, weights='distance'),
                "feat_fn": feat_pca,
                "feat_kwargs": {"n_components": n_pca},
            })

# --- 案C: 物理特徴量 + 単純モデル ---
for prep_name in ["SNV", "SNV+SG1d"]:
    for n_pca in [10, 20]:
        for n_bins in [5, 10]:
            # Combined + Ridge
            experiments.append({
                "name": f"C_Combined(pca{n_pca},bin{n_bins})+Ridge__{prep_name}",
                "group": "C: Combined",
                "prep_name": prep_name,
                "model_fn": lambda: Ridge(alpha=10),
                "feat_fn": feat_combined_compact,
                "feat_kwargs": {"n_pca": n_pca, "n_bins": n_bins},
            })
            # Combined + LightGBM
            experiments.append({
                "name": f"C_Combined(pca{n_pca},bin{n_bins})+LGB__{prep_name}",
                "group": "C: Combined",
                "prep_name": prep_name,
                "model_fn": lambda: LGBMRegressor(
                    n_estimators=500, learning_rate=0.05, num_leaves=15,
                    min_child_samples=20, verbosity=-1, random_state=42),
                "feat_fn": feat_combined_compact,
                "feat_kwargs": {"n_pca": n_pca, "n_bins": n_bins},
            })

# --- 案C追加: 水吸収帯特徴量のみ ---
for prep_name in ["raw", "SNV"]:
    experiments.append({
        "name": f"C_WaterBands+Ridge__{prep_name}",
        "group": "C: WaterBands",
        "prep_name": prep_name,
        "model_fn": lambda: Ridge(alpha=1),
        "feat_fn": feat_water_bands,
        "feat_kwargs": {},
    })
    experiments.append({
        "name": f"C_WaterBands+LGB__{prep_name}",
        "group": "C: WaterBands",
        "prep_name": prep_name,
        "model_fn": lambda: LGBMRegressor(
            n_estimators=300, learning_rate=0.05, num_leaves=15,
            min_child_samples=20, verbosity=-1, random_state=42),
        "feat_fn": feat_water_bands,
        "feat_kwargs": {},
    })

print(f"📊 全{len(experiments)}パターンを評価します")

# ============================================================
# 6. 全実験実行
# ============================================================
print("\n" + "=" * 70)
print("🚀 実験開始")
print("=" * 70)

all_results = []
pbar = tqdm(total=len(experiments), desc="実験進捗",
            bar_format='{l_bar}{bar:30}{r_bar}')

for exp in experiments:
    pbar.set_postfix_str(exp['name'][:40])
    t0 = time.time()

    try:
        prep_fn = PREPS.get(exp["prep_name"])
        rmse, fold_rmses, oof = evaluate(
            X_train_raw, y_train, species_train,
            exp["model_fn"],
            X_tr_raw_all=X_train_raw,
            prep_fn=prep_fn,
            feat_fn=exp["feat_fn"],
            feat_kwargs=exp["feat_kwargs"],
        )
        elapsed = time.time() - t0

        all_results.append({
            "name": exp["name"],
            "group": exp["group"],
            "rmse": rmse,
            "fold_std": np.std(fold_rmses),
            "fold_min": min(fold_rmses),
            "fold_max": max(fold_rmses),
            "time": elapsed,
        })

        if rmse < 25:
            tqdm.write(f"  ⭐ {exp['name']:50s} RMSE={rmse:7.2f} ({elapsed:.1f}s)")
        pbar.update(1)

    except Exception as e:
        tqdm.write(f"  ❌ {exp['name']:50s} Error: {e}")
        pbar.update(1)

pbar.close()

# ============================================================
# 7. 結果サマリー
# ============================================================
print("\n" + "=" * 70)
print("📋 結果サマリー（RMSE順 上位20）")
print("=" * 70)

res_df = pd.DataFrame(all_results).sort_values("rmse")
print(res_df.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("📋 グループ別ベスト")
print("=" * 70)
for group in res_df["group"].unique():
    best = res_df[res_df["group"] == group].iloc[0]
    print(f"  {group:20s} → RMSE={best['rmse']:7.2f}  {best['name']}")

# ============================================================
# 8. 上位5パターンのテスト予測を生成
# ============================================================
print("\n" + "=" * 70)
print("💾 上位パターンのテスト予測生成")
print("=" * 70)

top_experiments = res_df.head(5)["name"].tolist()
test_predictions = {}

for exp_name in tqdm(top_experiments, desc="テスト予測"):
    exp = [e for e in experiments if e["name"] == exp_name][0]
    prep_fn = PREPS.get(exp["prep_name"])

    # 前処理
    if prep_fn is not None:
        Xp_tr, Xp_te = prep_fn(X_train_raw, X_test_raw)
    else:
        Xp_tr, Xp_te = X_train_raw.copy(), X_test_raw.copy()

    # 特徴量生成
    if exp["feat_fn"] is not None:
        kwargs = exp["feat_kwargs"] or {}
        if "X_tr_raw" in exp["feat_fn"].__code__.co_varnames:
            F_tr, F_te, _ = exp["feat_fn"](Xp_tr, Xp_te, y_train, wavenumbers,
                                           X_train_raw, X_test_raw, **kwargs)
        else:
            F_tr, F_te, _ = exp["feat_fn"](Xp_tr, Xp_te, y_train, wavenumbers, **kwargs)
    else:
        F_tr, F_te = Xp_tr, Xp_te

    # 学習・予測
    model = exp["model_fn"]()
    if isinstance(model, (CatBoostRegressor,)):
        model.fit(F_tr, y_train, verbose=0)
    elif isinstance(model, LGBMRegressor):
        model.fit(F_tr, y_train)
    else:
        model.fit(F_tr, y_train)

    pred = np.clip(model.predict(F_te).ravel(), 0, None)
    test_predictions[exp_name] = pred
    tqdm.write(f"  ✅ {exp_name:50s} mean={pred.mean():.1f} [{pred.min():.1f}~{pred.max():.1f}]")

# アンサンブル（上位5の中央値）
if len(test_predictions) > 1:
    ens_pred = np.median(np.column_stack(list(test_predictions.values())), axis=1)
    test_predictions["ensemble_top5_median"] = ens_pred
    print(f"  ✅ {'ensemble_top5_median':50s} mean={ens_pred.mean():.1f} [{ens_pred.min():.1f}~{ens_pred.max():.1f}]")

# ============================================================
# 9. 提出ファイル生成
# ============================================================
print("\n" + "=" * 70)
print("💾 提出ファイル生成")
print("=" * 70)

today = datetime.now().strftime("%Y%m%d")
for tag, pred in test_predictions.items():
    safe_tag = tag.replace("(", "").replace(")", "").replace(",", "_").replace(" ", "_")
    filename = f"sub_{today}_{safe_tag}.csv"
    sub_df = pd.DataFrame({"sample number": test_df["sample number"], "含水率": pred})
    sub_df.to_csv(filename, index=False, encoding="cp932")

print(f"  {len(test_predictions)}個の提出ファイルを生成")

# ベスト単体 + アンサンブルを推奨
best_name = res_df.iloc[0]["name"]
print(f"\n🎯 推奨提出:")
print(f"  1位: {best_name} (GroupKFold RMSE={res_df.iloc[0]['rmse']:.2f})")
if "ensemble_top5_median" in test_predictions:
    print(f"  2位: ensemble_top5_median")

📂 データ読み込み...
  訓練: (1322, 1555), テスト: (550, 1555)
📊 全80パターンを評価します

🚀 実験開始


実験進捗:  36%|██████████▉                   | 29/80 [00:32<01:15,  1.49s/it, B_PCA(5)+LGB__SNV]        

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[26]	valid_0's l2: 707.71
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[37]	valid_0's l2: 895.286
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[111]	valid_0's l2: 2162.84
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[90]	valid_0's l2: 758.748


実験進捗:  38%|███████████▎                  | 30/80 [00:34<01:21,  1.63s/it, B_PCA(10)+Ridge__SNV]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[25]	valid_0's l2: 506.012


...

  ✅ C_Combined(pca10,bin5)+LGB__SNV+SG1d               mean=43.9 [4.4~187.7]


テスト予測:  80%|████████  | 4/5 [00:03<00:00,  1.29it/s]     

  ✅ C_Combined(pca10,bin10)+LGB__SNV+SG1d              mean=42.8 [7.2~176.3]


テスト予測: 100%|██████████| 5/5 [00:03<00:00,  1.42it/s]     

  ✅ A_Ridge(a=1000)__SNV                               mean=45.3 [0.0~162.9]
  ✅ ensemble_top5_median                               mean=45.3 [9.1~183.7]

💾 提出ファイル生成
  6個の提出ファイルを生成

🎯 推奨提出:
  1位: C_Combined(pca20,bin5)+LGB__SNV+SG1d (GroupKFold RMSE=23.50)
  2位: ensemble_top5_median


In [3]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold, GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from datetime import datetime
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])

X_train_raw = train_df[spectrum_cols].to_numpy(dtype=float)
y_train = train_df[target_col].values
species_train = train_df["species number"].values
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=float)

print(f"  訓練: {X_train_raw.shape}, テスト: {X_test_raw.shape}")

# ============================================================
# 2. 前処理
# ============================================================
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return (X - m) / s

def sg(X, window=17, poly=2, deriv=0):
    return savgol_filter(X, window, poly, deriv=deriv, axis=1)

# ============================================================
# 3. 評価: GroupKFold と LOGO の両方で確認
#    → Public Scoreとの相関を探る
# ============================================================
def eval_groupkfold(X_tr, y, species, model_fn, n_splits=5):
    cv = GroupKFold(n_splits=n_splits)
    oof = np.full(len(y), np.nan)
    for tr_idx, va_idx in cv.split(X_tr, y, groups=species):
        m = model_fn()
        if isinstance(m, PLSRegression):
            m.fit(X_tr[tr_idx], y[tr_idx])
        elif isinstance(m, LGBMRegressor):
            m.fit(X_tr[tr_idx], y[tr_idx], eval_set=[(X_tr[va_idx], y[va_idx])],
                  callbacks=[__import__('lightgbm').early_stopping(50),
                             __import__('lightgbm').log_evaluation(0)])
        else:
            m.fit(X_tr[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict(X_tr[va_idx]).ravel()
    valid = ~np.isnan(oof)
    return np.sqrt(mean_squared_error(y[valid], oof[valid]))

def eval_logo(X_tr, y, species, model_fn):
    """LOGO: 各樹種を1つずつ抜く → 樹種ごとのRMSE"""
    cv = LeaveOneGroupOut()
    results = {}
    for tr_idx, va_idx in cv.split(X_tr, y, groups=species):
        sp = species[va_idx][0]
        sp_name = train_df.loc[train_df['species number']==sp, '樹種'].iloc[0]
        m = model_fn()
        if isinstance(m, PLSRegression):
            m.fit(X_tr[tr_idx], y[tr_idx])
        elif isinstance(m, LGBMRegressor):
            m.fit(X_tr[tr_idx], y[tr_idx], eval_set=[(X_tr[va_idx], y[va_idx])],
                  callbacks=[__import__('lightgbm').early_stopping(50),
                             __import__('lightgbm').log_evaluation(0)])
        else:
            m.fit(X_tr[tr_idx], y[tr_idx])
        preds = m.predict(X_tr[va_idx]).ravel()
        rmse = np.sqrt(mean_squared_error(y[va_idx], preds))
        results[sp_name] = rmse
    # ベイスギ等の外れ値を除いたRMSEも計算
    all_rmses = list(results.values())
    trimmed = sorted(all_rmses)[:-2]  # 上位2つを除外
    return results, np.mean(all_rmses), np.mean(trimmed)

# ============================================================
# 4. 実験: シンプルなモデルを中心に
#    Public Score既知: 最初のLGBM=21
# ============================================================
print("\n" + "=" * 70)
print("🚀 実験開始: シンプルモデル × 前処理")
print("=" * 70)

configs = []

# --- PLS (成分数を振る) ---
for n in [2, 3, 5, 7, 10, 15, 20]:
    configs.append(("PLS", f"PLS({n})", "SNV+SG1d", 
                    lambda n_=n: PLSRegression(n_components=n_)))
    configs.append(("PLS", f"PLS({n})", "SNV",
                    lambda n_=n: PLSRegression(n_components=n_)))
    configs.append(("PLS", f"PLS({n})", "SG1d",
                    lambda n_=n: PLSRegression(n_components=n_)))
    configs.append(("PLS", f"PLS({n})", "raw",
                    lambda n_=n: PLSRegression(n_components=n_)))

# --- Ridge (正則化強度を振る) ---
for a in [0.01, 0.1, 1, 10, 100, 1000, 10000]:
    configs.append(("Ridge", f"Ridge(a={a})", "SNV",
                    lambda a_=a: Ridge(alpha=a_)))
    configs.append(("Ridge", f"Ridge(a={a})", "SNV+SG1d",
                    lambda a_=a: Ridge(alpha=a_)))

# --- LightGBM (シンプル設定、num_leavesを振る) ---
for nl in [7, 15, 31]:
    configs.append(("LGB", f"LGB(nl={nl})", "SNV+SG1d",
                    lambda nl_=nl: LGBMRegressor(
                        n_estimators=1000, learning_rate=0.05, num_leaves=nl_,
                        verbosity=-1, random_state=42)))
    configs.append(("LGB", f"LGB(nl={nl})", "SNV",
                    lambda nl_=nl: LGBMRegressor(
                        n_estimators=1000, learning_rate=0.05, num_leaves=nl_,
                        verbosity=-1, random_state=42)))
    configs.append(("LGB", f"LGB(nl={nl})", "raw",
                    lambda nl_=nl: LGBMRegressor(
                        n_estimators=1000, learning_rate=0.05, num_leaves=nl_,
                        verbosity=-1, random_state=42)))

print(f"  全{len(configs)}パターン")

# 前処理の適用
def apply_prep(name, X_tr, X_te):
    if name == "raw":
        return X_tr.copy(), X_te.copy()
    elif name == "SNV":
        return snv(X_tr), snv(X_te)
    elif name == "SNV+SG1d":
        return sg(snv(X_tr), 17, 2, 1), sg(snv(X_te), 17, 2, 1)
    elif name == "SG1d":
        return sg(X_tr, 17, 2, 1), sg(X_te, 17, 2, 1)

# 実行
all_results = []
pbar = tqdm(total=len(configs), desc="実験", bar_format='{l_bar}{bar:30}{r_bar}')

for group, model_name, prep_name, model_fn in configs:
    pbar.set_postfix_str(f"{model_name}__{prep_name}"[:40])
    t0 = time.time()

    Xp_tr, Xp_te = apply_prep(prep_name, X_train_raw, X_test_raw)
    gkf_rmse = eval_groupkfold(Xp_tr, y_train, species_train, model_fn)
    elapsed = time.time() - t0

    # テスト予測も同時に生成
    model = model_fn()
    if isinstance(model, LGBMRegressor):
        model.fit(Xp_tr, y_train)
    else:
        model.fit(Xp_tr, y_train)
    test_pred = np.clip(model.predict(Xp_te).ravel(), 0, None)

    all_results.append({
        "group": group, "model": model_name, "prep": prep_name,
        "gkf_rmse": gkf_rmse, "time": elapsed,
        "test_mean": test_pred.mean(), "test_std": test_pred.std(),
        "test_pred": test_pred,
    })

    if gkf_rmse < 30:
        tqdm.write(f"  ⭐ {model_name:15s} {prep_name:10s} GKF={gkf_rmse:6.2f} "
                   f"test_mean={test_pred.mean():5.1f} ⏱{elapsed:.1f}s")
    pbar.update(1)

pbar.close()

# ============================================================
# 5. 結果サマリー
# ============================================================
res_df = pd.DataFrame([{k:v for k,v in r.items() if k != "test_pred"} 
                        for r in all_results]).sort_values("gkf_rmse")

print("\n" + "=" * 70)
print("📋 上位20パターン")
print("=" * 70)
print(res_df.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("📋 モデル種別ごとのベスト")
print("=" * 70)
for group in ["PLS", "Ridge", "LGB"]:
    sub = res_df[res_df["group"] == group]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"  {group:8s} → GKF_RMSE={best['gkf_rmse']:6.2f}  "
              f"{best['model']} + {best['prep']}")

# ============================================================
# 6. 上位モデルのLOGO詳細分析（どの樹種が苦手か比較）
# ============================================================
print("\n" + "=" * 70)
print("🔍 上位3モデルのLOGO分析")
print("=" * 70)

top3 = res_df.head(3)
for _, row in top3.iterrows():
    exp = [r for r in all_results if r["model"] == row["model"] and r["prep"] == row["prep"]][0]
    model_fn = [c[3] for c in configs if c[1] == row["model"] and c[2] == row["prep"]][0]
    
    Xp_tr, _ = apply_prep(row["prep"], X_train_raw, X_test_raw)
    species_rmse, mean_rmse, trimmed_rmse = eval_logo(Xp_tr, y_train, species_train, model_fn)
    
    print(f"\n  {row['model']} + {row['prep']} (GKF={row['gkf_rmse']:.2f})")
    print(f"  LOGO平均: {mean_rmse:.2f}, トリム平均(上位2除外): {trimmed_rmse:.2f}")
    for sp, rmse in sorted(species_rmse.items(), key=lambda x: x[1], reverse=True):
        marker = "🔴" if rmse > 30 else "🟡" if rmse > 20 else "🟢"
        print(f"    {marker} {sp:15s} RMSE={rmse:6.1f}")

# ============================================================
# 7. 提出ファイル生成（複数候補）
# ============================================================
print("\n" + "=" * 70)
print("💾 提出ファイル生成")
print("=" * 70)

today = datetime.now().strftime("%Y%m%d")

# GroupKFold上位1位
best1 = all_results[res_df.index[0]]
# PLS ベスト
pls_best_idx = res_df[res_df["group"]=="PLS"].index[0]
pls_best = all_results[pls_best_idx]
# Ridge ベスト
ridge_best_idx = res_df[res_df["group"]=="Ridge"].index[0]
ridge_best = all_results[ridge_best_idx]
# LGB ベスト
lgb_best_idx = res_df[res_df["group"]=="LGB"].index[0]
lgb_best = all_results[lgb_best_idx]

# 各モデル種別のベスト予測でアンサンブル
ens_pred = np.mean([pls_best["test_pred"], ridge_best["test_pred"], lgb_best["test_pred"]], axis=0)
ens_pred = np.clip(ens_pred, 0, None)

candidates = {
    f"GKF_best__{best1['model']}__{best1['prep']}": best1["test_pred"],
    f"PLS_best__{pls_best['model']}__{pls_best['prep']}": pls_best["test_pred"],
    f"Ridge_best__{ridge_best['model']}__{ridge_best['prep']}": ridge_best["test_pred"],
    f"LGB_best__{lgb_best['model']}__{lgb_best['prep']}": lgb_best["test_pred"],
    "ensemble_PLS_Ridge_LGB": ens_pred,
}

for tag, pred in candidates.items():
    safe_tag = tag.replace("(", "").replace(")", "").replace("=", "").replace(" ", "")
    filename = f"sub_{today}_{safe_tag}.csv"
    sub_df = pd.DataFrame({"sample number": test_df["sample number"], "含水率": pred})
    sub_df.to_csv(filename, index=False, encoding="cp932")
    print(f"  📝 {filename}")
    print(f"     mean={pred.mean():.1f} [{pred.min():.1f}~{pred.max():.1f}]")

print(f"""
{'='*70}
📌 提出の考え方
{'='*70}
前回の教訓: GroupKFold RMSE が低い ≠ Public Score が良い

今回の推奨（1回だけ提出）:
  → PLS ベスト または ensemble_PLS_Ridge_LGB
  → 理由: 線形モデルは外挿に強い
  → 残りの提出枠は温存

前回Public Score:
  最初のLGBM（デフォルト）: 21
  CatBoost+LGB(Optuna):    24.4
  
→ 今回の結果で改善できるか確認
""")

📂 データ読み込み...
  訓練: (1322, 1555), テスト: (550, 1555)

🚀 実験開始: シンプルモデル × 前処理
  全51パターン


実験:  12%|███▌                          | 6/51 [00:02<00:14,  3.03it/s, PLS(3)__SG1d]    

  ⭐ PLS(3)          SNV        GKF= 27.85 test_mean= 40.5 ⏱0.2s


実験:  76%|██████████████████████▉       | 39/51 [00:30<00:16,  1.34s/it, Ridge(a=1000)__SNV+SG1d]

  ⭐ Ridge(a=1000)   SNV        GKF= 26.98 test_mean= 45.3 ⏱1.0s


...

  ⭐ LGB(nl=31)      SNV+SG1d   GKF= 23.90 test_mean= 43.9 ⏱16.6s
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[30]	valid_0's l2: 332.674
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[66]	valid_0's l2: 338.802
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[32]	valid_0's l2: 2870.83
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[83]	valid_0's l2: 671.723
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[779]	valid_0's l2: 224.827


実験:  98%|█████████████████████████████▍| 50/51 [03:30<00:29, 29.50s/it, LGB(nl=31)__raw]  

  ⭐ LGB(nl=31)      SNV        GKF= 29.63 test_mean= 44.3 ⏱24.9s
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[69]	valid_0's l2: 129.723
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[127]	valid_0's l2: 1275
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[737]	valid_0's l2: 2613.23
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's l2: 2235.3
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[67]	valid_0's l2: 137.393


実験: 100%|██████████████████████████████| 51/51 [04:14<00:00,  4.98s/it, LGB(nl=31)__raw]



📋 上位20パターン
group          model     prep  gkf_rmse      time  test_mean  test_std
  LGB      LGB(nl=7) SNV+SG1d 23.140591  5.348087  44.291361 42.782073
  LGB     LGB(nl=15) SNV+SG1d 23.668928  9.086686  42.668750 42.529686
  LGB     LGB(nl=31) SNV+SG1d 23.896414 16.628947  43.874367 42.515616
Ridge  Ridge(a=1000)      SNV 26.978041  1.032231  45.326179 41.486693
  PLS         PLS(3)      SNV 27.847695  0.212251  40.527973 44.662373
  LGB      LGB(nl=7)      SNV 29.288780  3.871975  48.735014 39.659759
  LGB     LGB(nl=15)      SNV 29.420300  9.367384  46.538059 41.262461
  LGB     LGB(nl=31)      SNV 29.625913 24.887610  44.252277 43.534138
Ridge Ridge(a=10000)      SNV 32.304232  1.065877  48.180470 27.035631
  PLS         PLS(2)      SNV 32.456308  0.220263  40.568571 44.450066
  PLS         PLS(3)      raw 34.265269  0.182463  49.603422 33.979199
  LGB     LGB(nl=15)      raw 35.162638 11.824119  57.913114 37.429541
  PLS         PLS(2)     SG1d 35.296145  0.413508  48.457039 37.9

In [4]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from datetime import datetime
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. データ読み込み
# ============================================================
print("📂 データ読み込み...")
train_df = pd.read_csv("data/train.csv", encoding="cp932")
test_df = pd.read_csv("data/test.csv", encoding="cp932")

target_col = "含水率"
meta_cols = ["sample number", "species number", "樹種", "含水率"]
spectrum_cols = [c for c in train_df.columns if c not in meta_cols]
wavenumbers = np.array([float(c) for c in spectrum_cols])

X_train_raw = train_df[spectrum_cols].to_numpy(dtype=float)
y_train = train_df[target_col].values
species_train = train_df["species number"].values
X_test_raw = test_df[spectrum_cols].to_numpy(dtype=float)

print(f"  訓練: {X_train_raw.shape}, テスト: {X_test_raw.shape}")

# ============================================================
# 2. 前処理
# ============================================================
def snv(X):
    m = X.mean(axis=1, keepdims=True)
    s = X.std(axis=1, keepdims=True)
    s[s == 0] = 1.0
    return (X - m) / s

# ============================================================
# 3. LOGO評価（トリム平均で評価 = Public Scoreの近似）
# ============================================================
def eval_logo_trimmed(X_tr, y, species, model_fn, n_trim=2):
    """LOGOのトリム平均（上位n_trimを除外）"""
    cv = LeaveOneGroupOut()
    sp_rmses = {}
    for tr_idx, va_idx in cv.split(X_tr, y, groups=species):
        sp = species[va_idx][0]
        m = model_fn()
        if isinstance(m, PLSRegression):
            m.fit(X_tr[tr_idx], y[tr_idx])
        elif isinstance(m, LGBMRegressor):
            m.fit(X_tr[tr_idx], y[tr_idx],
                  eval_set=[(X_tr[va_idx], y[va_idx])],
                  callbacks=[early_stopping(50), log_evaluation(0)])
        else:
            m.fit(X_tr[tr_idx], y[tr_idx])
        preds = m.predict(X_tr[va_idx]).ravel()
        sp_rmses[sp] = np.sqrt(mean_squared_error(y[va_idx], preds))
    
    all_rmses = list(sp_rmses.values())
    trimmed = sorted(all_rmses)[:-n_trim]
    return np.mean(all_rmses), np.mean(trimmed), sp_rmses

# ============================================================
# 4. 実験1: SGウィンドウサイズの影響
# ============================================================
print("\n" + "=" * 70)
print("🔬 実験1: SG1dウィンドウサイズの影響")
print("=" * 70)

sg_results = []
windows = [5, 7, 9, 11, 13, 15, 17, 21, 25, 31]

pbar = tqdm(total=len(windows) * 3, desc="SGウィンドウ探索",
            bar_format='{l_bar}{bar:30}{r_bar}')

for w in windows:
    if w < 3:
        continue
    X_tr_sg = savgol_filter(snv(X_train_raw), w, 2, deriv=1, axis=1)

    for model_name, model_fn in [
        ("PLS(5)", lambda: PLSRegression(n_components=5)),
        ("Ridge(1000)", lambda: Ridge(alpha=1000)),
        ("LGB(nl=7)", lambda: LGBMRegressor(
            n_estimators=1000, learning_rate=0.05, num_leaves=7,
            verbosity=-1, random_state=42)),
    ]:
        pbar.set_postfix_str(f"w={w}, {model_name}")
        t0 = time.time()
        logo_mean, logo_trim, _ = eval_logo_trimmed(
            X_tr_sg, y_train, species_train, model_fn)
        elapsed = time.time() - t0

        sg_results.append({
            "window": w, "model": model_name,
            "logo_mean": logo_mean, "logo_trim": logo_trim,
            "time": elapsed,
        })
        tqdm.write(f"  w={w:2d} {model_name:15s} LOGO_mean={logo_mean:6.2f} "
                   f"LOGO_trim={logo_trim:6.2f} ⏱{elapsed:.1f}s")
        pbar.update(1)

pbar.close()

sg_df = pd.DataFrame(sg_results)
print("\n--- SGウィンドウ × モデル: LOGOトリム平均 ---")
pivot = sg_df.pivot_table(values="logo_trim", index="window", columns="model")
print(pivot.round(2).to_string())

# ============================================================
# 5. 実験2: 前処理の組み合わせ（SNVなし/ありも含む）
# ============================================================
print("\n" + "=" * 70)
print("🔬 実験2: 前処理パターン × LOGO トリム平均")
print("=" * 70)

# ベストwindowを使用
best_w_row = sg_df.loc[sg_df["logo_trim"].idxmin()]
best_w = int(best_w_row["window"])
print(f"  ベストSGウィンドウ: {best_w}")

prep_results = []

prep_patterns = {
    "raw": lambda Xtr, Xte: (Xtr.copy(), Xte.copy()),
    "SNV": lambda Xtr, Xte: (snv(Xtr), snv(Xte)),
    f"SG1d(w={best_w})": lambda Xtr, Xte: (
        savgol_filter(Xtr, best_w, 2, deriv=1, axis=1),
        savgol_filter(Xte, best_w, 2, deriv=1, axis=1)),
    f"SNV+SG1d(w={best_w})": lambda Xtr, Xte: (
        savgol_filter(snv(Xtr), best_w, 2, deriv=1, axis=1),
        savgol_filter(snv(Xte), best_w, 2, deriv=1, axis=1)),
    "SNV+SG1d(w=11)": lambda Xtr, Xte: (
        savgol_filter(snv(Xtr), 11, 2, deriv=1, axis=1),
        savgol_filter(snv(Xte), 11, 2, deriv=1, axis=1)),
    "SG1d(w=11)": lambda Xtr, Xte: (
        savgol_filter(Xtr, 11, 2, deriv=1, axis=1),
        savgol_filter(Xte, 11, 2, deriv=1, axis=1)),
}

model_configs = {
    "PLS(3)":      lambda: PLSRegression(n_components=3),
    "PLS(5)":      lambda: PLSRegression(n_components=5),
    "PLS(7)":      lambda: PLSRegression(n_components=7),
    "Ridge(100)":  lambda: Ridge(alpha=100),
    "Ridge(1000)": lambda: Ridge(alpha=1000),
    "LGB(nl=7)":   lambda: LGBMRegressor(
        n_estimators=1000, learning_rate=0.05, num_leaves=7,
        verbosity=-1, random_state=42),
}

total = len(prep_patterns) * len(model_configs)
pbar = tqdm(total=total, desc="前処理×モデル", bar_format='{l_bar}{bar:30}{r_bar}')

for prep_name, prep_fn in prep_patterns.items():
    Xp_tr, Xp_te = prep_fn(X_train_raw, X_test_raw)

    for model_name, model_fn in model_configs.items():
        pbar.set_postfix_str(f"{prep_name[:15]}+{model_name}")
        t0 = time.time()

        logo_mean, logo_trim, sp_rmses = eval_logo_trimmed(
            Xp_tr, y_train, species_train, model_fn)

        # テスト予測
        model = model_fn()
        if isinstance(model, LGBMRegressor):
            model.fit(Xp_tr, y_train)
        else:
            model.fit(Xp_tr, y_train)
        test_pred = np.clip(model.predict(Xp_te).ravel(), 0, None)
        elapsed = time.time() - t0

        prep_results.append({
            "prep": prep_name, "model": model_name,
            "logo_mean": logo_mean, "logo_trim": logo_trim,
            "test_mean": test_pred.mean(), "test_std": test_pred.std(),
            "time": elapsed, "test_pred": test_pred,
        })

        if logo_trim < 16:
            tqdm.write(f"  ⭐ {prep_name:20s} {model_name:15s} "
                       f"LOGO_trim={logo_trim:5.2f} test_mean={test_pred.mean():5.1f}")
        pbar.update(1)

pbar.close()

# ============================================================
# 6. 結果サマリー（LOGOトリム平均で順位付け）
# ============================================================
prep_df = pd.DataFrame([{k: v for k, v in r.items() if k != "test_pred"}
                         for r in prep_results]).sort_values("logo_trim")

print("\n" + "=" * 70)
print("📋 上位20パターン（LOGOトリム平均順）")
print("=" * 70)
print(prep_df.head(20).to_string(index=False))

print("\n" + "=" * 70)
print("📋 モデル種別ベスト（LOGOトリム平均）")
print("=" * 70)
for model_name in ["PLS(3)", "PLS(5)", "PLS(7)", "Ridge(100)", "Ridge(1000)", "LGB(nl=7)"]:
    sub = prep_df[prep_df["model"] == model_name]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"  {model_name:15s} → trim={best['logo_trim']:5.2f}  {best['prep']}")

print("\n" + "=" * 70)
print("📋 前処理ベスト（LOGOトリム平均）")
print("=" * 70)
for prep_name in prep_patterns.keys():
    sub = prep_df[prep_df["prep"] == prep_name]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"  {prep_name:25s} → trim={best['logo_trim']:5.2f}  {best['model']}")

# ============================================================
# 7. 提出候補生成
# ============================================================
print("\n" + "=" * 70)
print("💾 提出ファイル生成")
print("=" * 70)

today = datetime.now().strftime("%Y%m%d")

# LOGOトリム平均 上位5つの個別予測
top5_names = []
for i, row in prep_df.head(5).iterrows():
    exp = [r for r in prep_results 
           if r["prep"] == row["prep"] and r["model"] == row["model"]][0]
    tag = f"{row['model']}__{row['prep']}".replace("(","").replace(")","").replace("=","").replace(" ","")
    filename = f"sub_{today}_top{len(top5_names)+1}_{tag}.csv"
    sub_df = pd.DataFrame({"sample number": test_df["sample number"], "含水率": exp["test_pred"]})
    sub_df.to_csv(filename, index=False, encoding="cp932")
    top5_names.append((filename, exp["test_pred"], row["logo_trim"]))
    print(f"  📝 {filename}")
    print(f"     trim={row['logo_trim']:.2f} mean={exp['test_pred'].mean():.1f} "
          f"[{exp['test_pred'].min():.1f}~{exp['test_pred'].max():.1f}]")

# 上位5のアンサンブル（平均）
ens_mean = np.mean([t[1] for t in top5_names], axis=0)
ens_mean = np.clip(ens_mean, 0, None)
filename_ens = f"sub_{today}_ensemble_top5_mean.csv"
sub_df = pd.DataFrame({"sample number": test_df["sample number"], "含水率": ens_mean})
sub_df.to_csv(filename_ens, index=False, encoding="cp932")
print(f"\n  📝 {filename_ens}")
print(f"     mean={ens_mean.mean():.1f} [{ens_mean.min():.1f}~{ens_mean.max():.1f}]")

# 上位5のアンサンブル（中央値）
ens_med = np.median([t[1] for t in top5_names], axis=0)
ens_med = np.clip(ens_med, 0, None)
filename_ens2 = f"sub_{today}_ensemble_top5_median.csv"
sub_df = pd.DataFrame({"sample number": test_df["sample number"], "含水率": ens_med})
sub_df.to_csv(filename_ens2, index=False, encoding="cp932")
print(f"  📝 {filename_ens2}")
print(f"     mean={ens_med.mean():.1f} [{ens_med.min():.1f}~{ens_med.max():.1f}]")

print(f"""
{'='*70}
📌 提出戦略
{'='*70}
評価指標の使い分け:
  GroupKFold RMSE → ベイスギ等の外れ値に引っ張られる → Public Score と逆転
  LOGO トリム平均 → 外れ値を除外 → Public Score の良い近似（仮説）

既知の Public Score:
  最初のLGBM: 21（SNV+SG1d w=11, species OHE付き）
  CatBoost+LGB: 24.4（SG2d, species なし）

推奨提出（1回だけ）:
  LOGOトリム平均が最小のパターン
  → 理由: テストの6種がベイスギ的な外れ種を含まないなら
           トリム平均が最もPublic Scoreに近い

これで Public Score が改善すれば:
  → LOGO トリム平均 が正しい指標だと確認
  → 以降はこの指標で最適化

改善しなければ:
  → 別の問題がある（提出フォーマット、学習方法等）
  → チームメンバーの手法を確認するしかない
""")

📂 データ読み込み...
  訓練: (1322, 1555), テスト: (550, 1555)

🔬 実験1: SG1dウィンドウサイズの影響


SGウィンドウ探索:   3%|█                             | 1/30 [00:01<00:40,  1.40s/it, w=5, Ridge(1000)]

  w= 5 PLS(5)          LOGO_mean= 26.65 LOGO_trim= 17.00 ⏱1.4s


SGウィンドウ探索:   7%|██                            | 2/30 [00:04<01:11,  2.54s/it, w=5, LGB(nl=7)]         

  w= 5 Ridge(1000)     LOGO_mean= 46.77 LOGO_trim= 41.16 ⏱3.3s
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[72]	valid_0's l2: 268.997
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[174]	valid_0's l2: 379.931
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[62]	valid_0's l2: 317.217
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[84]	valid_0's l2: 155.357
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[278]	valid_0's l2: 96.3364
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[27]	valid_0's l2: 455.633
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[63]	valid_0's l2: 29.5597
Training until validation scores don't improve for 50 rounds
Early stoppin

...

  ⭐ SNV+SG1d(w=11)       LGB(nl=7)       LOGO_trim=14.03 test_mean= 44.6


前処理×モデル:  97%|█████████████████████████████▏| 35/36 [02:15<00:03,  3.07s/it, SG1d(w=11)+LGB(nl=7)]  

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[320]	valid_0's l2: 281.398
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[361]	valid_0's l2: 171.858
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[29]	valid_0's l2: 323.098
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[110]	valid_0's l2: 167.31
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[76]	valid_0's l2: 109.502
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[28]	valid_0's l2: 512.068
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[128]	valid_0's l2: 34.0104
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[42]	valid_0's l2: 104.562
Training unti

前処理×モデル: 100%|██████████████████████████████| 36/36 [02:35<00:00,  4.31s/it, SG1d(w=11)+LGB(nl=7)]      

  ⭐ SG1d(w=11)           LGB(nl=7)       LOGO_trim=13.27 test_mean= 47.3

📋 上位20パターン（LOGOトリム平均順）
          prep       model  logo_mean  logo_trim  test_mean  test_std      time
 SNV+SG1d(w=5)   LGB(nl=7)  16.659897  12.584886  44.350243 44.531991 18.746825
    SG1d(w=11)   LGB(nl=7)  18.142282  13.271606  47.348110 42.679110 19.539367
     SG1d(w=5)   LGB(nl=7)  17.831661  13.453457  45.102081 43.964966 16.668098
           SNV   LGB(nl=7)  19.731463  13.715830  48.735014 39.659759 15.125306
SNV+SG1d(w=11)   LGB(nl=7)  18.260316  14.031829  44.600545 41.410717 18.172577
 SNV+SG1d(w=5)      PLS(7)  23.113196  14.552269  64.714262 40.888477  0.933317
           SNV Ridge(1000)  20.421115  14.765505  45.326179 41.486693  2.997637
SNV+SG1d(w=11)      PLS(3)  23.143793  15.075085  51.333599 43.840751  0.623340
 SNV+SG1d(w=5)      PLS(3)  22.395948  15.207996  53.675929 43.770214  0.566128
           SNV      PLS(3)  21.352559  15.486922  40.527973 44.662373  0.609545
           SNV  Ridge(1